# L15b Lab: The Library Solve, Explicit Euler, and Runge–Kutta 4
A pulse of an inducer, a chemical added to the medium, can switch a gene circuit from a low state to a high state that stays after the pulse ends. We compute that response by solving the material balances of the circuit forward in time from a known starting state, which is an initial value problem. How does a library solver handle the jumps in the inducer concentration, and how do fixed-step methods, which take equal steps of a size we choose, compare with that solver?

We'll explore these questions on the memory circuit from [the example on explicit methods](../L15a/CHEME-5800-L15a-Example-ExplicitMethods-Fall-2026.ipynb). You will write the code for a library solve, explicit Euler, and classical fourth-order Runge–Kutta (RK4).

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Solve an initial value problem with a library solver:__ Call the Tsit5 solver with a stop at each jump of the input, so that a step ends there. Compare a short pulse solved with and without the stops.
> * __Implement explicit Euler and RK4:__ Write the code for one step of each method and for a loop of equal steps. Check each function against the course package, and compare Euler with the library solve.
> * __Measure how the error changes with the step:__ Run both methods on several grids where the input stays constant. Compare how fast the largest error falls when the step is halved.

Let's get started!

___

## Setup, Data, and Prerequisites
First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [ ]:
# Load the local setup and choose the plot theme -
include(joinpath(@__DIR__, "Include.jl")); # load the shared course environment, this lab's code, and the lab's imports

# Choose one theme, then rerun the plotting cells after a change -
theme(:default); # light background
# theme(:dark); # dark background

[The `balances!(...)` function](../../../code/src/GeneNetworks.jl) of the course package computes the rates of change of a circuit. The state is the concentration vector: the messenger RNA (mRNA) concentrations, followed by the protein concentrations.

In the [memory circuit](config/memory.toml), the inducer activates protein X. Protein X activates genes Y and Z, whose proteins activate each other's genes.

Let's choose the settings:

In [ ]:
# the time grid -
final_time = 480.0; # the end of every run from the low steady state (min); each run stops at the whole step nearest to it
Δt = 4.0; # the step of the fixed-step runs through the long pulse, in Tasks 2 and 3 (min)
Δt_save = 1.0; # the time between the states that the library solve saves in Task 1 (min)

# the inducer pulses. The inducer I starts at its baseline, the value initial under [inputs.I] in the config file.
# At pulse_start, I changes to pulse_concentration. At the end of the pulse, I returns to the baseline -
pulse_concentration = 4.0; # the inducer concentration during a pulse (µM)
pulse_start = 40.0; # the time at which a pulse starts (min)
pulse_length = 72.0; # how long the pulse of Tasks 2 and 3 lasts (min)
short_pulse_length = 20.0; # how long the short pulse of Task 1 lasts (min)

# the library solver -
relative_tolerance = 1e-10; # the error that Tsit5 allows in each step, relative to each concentration
absolute_tolerance = 1e-12; # the error that Tsit5 allows near a concentration of zero (nmol/gDW)

# the step study of Task 3, on a window that starts at the end of the pulse -
window_length = 48.0; # the length of the window (min)
step_counts = [32, 64, 128]; # the number of steps on each grid of the window; doubling a count halves the step

The circuit starts at its low steady state, with the inducer at its baseline. At a steady state, every rate of change is zero. [The `steady_state(...)` function](../../../code/src/GeneNetworks.jl) finds that state with a search that starts at zero mRNA and protein.

Let's build the memory circuit and find its low steady state:

In [ ]:
(model, x₀, protein_y) = let

    # initialize -
    path = joinpath(@__DIR__, "config", "memory.toml"); # the config file of the memory circuit
    t_start = 0.0; # the time of the steady state (min); the inducer is at its baseline

    # build the model from the config file -
    model = build(MyGeneNetworkModel, path); # read the file, and build the model of the circuit
    N = length(model.genes); # the number of genes; the state is [m₁, ..., m_N, p₁, ..., p_N]
    protein_y = N + model.index["Y"]; # the entry of protein Y in the state: the N mRNA entries come first

    # the low steady state, with the inducer at its baseline -
    guess = zeros(2 * N); # start the search at zero mRNA and protein (nmol/gDW)
    solution = steady_state(model; guess = guess, t = t_start); # the state at which every balance is zero
    if solution.converged == false # the search did not reach a steady state
        error("No steady state was found. Undo the last change to the config file, and rerun the notebook."); # stop here
    end

    (model, solution.x, protein_y); # return
end;

Task 1 uses the short pulse, set by `short_pulse_length`. Tasks 2 and 3 use the long pulse, set by `pulse_length`.

For the Task 1 runs, we round `final_time` to the nearest multiple of `Δt_save`. For the Task 2 runs, we round `final_time` and the edges of the long pulse to the nearest multiples of `Δt`, so that a step ends at each edge.

Let's set both pulses and check that each one fits inside its runs:

In [ ]:
(short_pulse, long_pulse, t_final_save, t_final_step) = let

    # initialize -
    baseline = model.inputs["I"].initial; # the baseline inducer concentration from the config file (µM)
    t_final_save = round(final_time / Δt_save) * Δt_save; # the end of the Task 1 runs: the whole save step nearest to final_time (min)
    t_final_step = round(final_time / Δt) * Δt; # the end of the Task 2 runs: the whole step nearest to final_time (min)

    # the short pulse of Task 1 keeps its edges, because the solver stops can land on any time -
    short_pulse = (start = pulse_start, stop = pulse_start + short_pulse_length, # the edges of the short pulse (min)
        concentration = pulse_concentration, baseline = baseline); # the inducer during and after the pulse (µM)

    # the pulse of Tasks 2 and 3, with its edges rounded to whole steps of Δt -
    long_pulse = (start = round(pulse_start / Δt) * Δt, stop = round((pulse_start + pulse_length) / Δt) * Δt, # the rounded edges (min)
        concentration = pulse_concentration, baseline = baseline); # the inducer during and after the pulse (µM)

    # stop when a pulse does not fit inside its runs -
    if (0.0 < short_pulse.start < short_pulse.stop < t_final_save) == false # the short pulse must start after zero and end before the runs end
        error("The short pulse does not fit inside the Task 1 runs. Change pulse_start, short_pulse_length, or final_time, and rerun the notebook."); # stop here
    end
    if (0.0 < long_pulse.start < long_pulse.stop < t_final_step) == false # the rounded pulse must last at least one step and end before the runs end
        error("The pulse does not fit inside the Task 2 runs. Change pulse_start, pulse_length, or final_time, and rerun the notebook."); # stop here
    end

    (short_pulse, long_pulse, t_final_save, t_final_step); # return
end;

___
## Task 1: The Library Solve
In this task, we ask what happens when a library solver does not know where the inducer jumps. [The `Tsit5` solver](https://docs.sciml.ai/OrdinaryDiffEq/stable/explicit/Tsit5/) picks the size of each step itself, so that the estimated error of each step stays below the tolerances. The [`tstops` option](https://docs.sciml.ai/DiffEqDocs/stable/basics/common_solver_opts/#Output-Control) gives the solver a list of stops, times at which the solver must end a step.

Open [`src/Compute.jl`](src/Compute.jl) and complete [the `mysolve(...)` function](src/Compute.jl), where a TODO comment with a Hint marks each line you write. Save the file, and rerun the setup cell to reload your edits. The untouched function throws an error.

[The `mysolve(...)` function](src/Compute.jl) returns the saved times `t`, the states `x` with one column per saved time, and the number of rate evaluations, the calls of [the `balances!(...)` function](../../../code/src/GeneNetworks.jl). Let's check our function against a direct call of [the `solve(...)` function of the OrdinaryDiffEq package](https://docs.sciml.ai/DiffEqDocs/stable/getting_started/) with the same settings:

In [ ]:
let

    # initialize -
    t_start = 0.0; # the time at which the runs start (min)
    edges = [short_pulse.start, short_pulse.stop]; # the times at which the inducer jumps (min)
    set_input!(model, "I", edges, [short_pulse.concentration, short_pulse.baseline]); # the short pulse, then the baseline again

    # our solve, and a direct call of the library with the same settings -
    ours = L15bMethods.mysolve(balances!, x₀, model; t_start = t_start, t_final = t_final_save, # the balances, the starting state, and the time span
        stops = edges, Δt_save = Δt_save, reltol = relative_tolerance, abstol = absolute_tolerance); # the stops, the save step, and the tolerances
    problem = ODEProblem(balances!, copy(x₀), (t_start, t_final_save), model); # the same problem
    direct = OrdinaryDiffEq.solve(problem, Tsit5(); tstops = edges, saveat = ours.t, # the same stops and save times
        reltol = relative_tolerance, abstol = absolute_tolerance); # the same tolerances

    # the saved times and the size of the states -
    number_of_saves = round(Int, (t_final_save - t_start) / Δt_save); # the number of save steps
    @test length(ours.t) == number_of_saves + 1; # one saved time per save step, plus the start
    @test ours.t[1] == t_start; # the first saved time is the start of the run
    @test ours.t[end] == t_final_save; # the last saved time is the end of the run
    @test ours.t == direct.t; # the same saved times
    @test size(ours.x) == (length(x₀), length(direct.t)); # one column per saved time

    # compare the saved states, one time at a time -
    same_states = true; # becomes false at the first saved state that differs
    for k ∈ 1:length(direct.t) # each saved time
        if ours.x[:, k] != direct.u[k] # column k must equal the state at the k-th saved time
            same_states = false; # a saved state differs
        end
    end
    @test same_states; # the same saved states
    @test ours.evaluations == direct.stats.nf; # the same number of rate evaluations
end

Let's solve the short pulse with the stops and without them, and plot protein Y from both runs:

In [ ]:
(with_stops, without_stops) = let

    # initialize -
    t_start = 0.0; # the time at which the runs start (min)
    edges = [short_pulse.start, short_pulse.stop]; # the times at which the inducer jumps (min)
    set_input!(model, "I", edges, [short_pulse.concentration, short_pulse.baseline]); # the short pulse, then the baseline again

    # the same solve, with a stop at each pulse edge and with no stops -
    with_stops = L15bMethods.mysolve(balances!, x₀, model; t_start = t_start, t_final = t_final_save, # the balances, the starting state, and the time span
        stops = edges, Δt_save = Δt_save, reltol = relative_tolerance, abstol = absolute_tolerance); # a step ends at each pulse edge
    without_stops = L15bMethods.mysolve(balances!, x₀, model; t_start = t_start, t_final = t_final_save, # the same problem
        stops = Float64[], Δt_save = Δt_save, reltol = relative_tolerance, abstol = absolute_tolerance); # the solver picks every step freely

    (with_stops, without_stops); # return
end;

plot_stops(with_stops, without_stops, model, short_pulse) # protein Y from both runs, below the inducer

Both runs start at the low steady state, where every rate is zero. Where does the run without the stops leave the run with the stops? Why can a solver that picks its own steps pass over the whole pulse?

___
## Task 2: Explicit Euler
In this task, we ask whether explicit Euler at the step `Δt` follows the library solve through the long pulse. Explicit Euler moves along the slope at the start of each step for the whole step.

Complete [the `mystep(...)` function](src/Compute.jl) for explicit Euler, and complete [the `myintegrate(...)` function](src/Compute.jl). Save the file, and rerun the setup cell.

Julia runs the version of [the `mystep(...)` function](src/Compute.jl) that matches the type of its first argument, [the `ExplicitEulerMethod` type](../../../code/src/FixedStepMethods.jl) or [the `RungeKutta4Method` type](../../../code/src/FixedStepMethods.jl). One loop in [the `myintegrate(...)` function](src/Compute.jl) then runs both methods. That function returns `t`, `x`, and the rate evaluations as [the `mysolve(...)` function](src/Compute.jl) does.

Let's check our functions against [the `advance(...)` function](../../../code/src/FixedStepMethods.jl) and [the `integrate(...)` function](../../../code/src/FixedStepMethods.jl) of the course package:

In [ ]:
let

    # initialize -
    t_start = 0.0; # the time at which the runs start (min)
    round_off = 1e-10; # the relative difference that round-off can cause between two correct runs
    edges = [long_pulse.start, long_pulse.stop]; # the pulse edges, on the grid of Δt (min)
    set_input!(model, "I", edges, [long_pulse.concentration, long_pulse.baseline]); # the long pulse, then the baseline again

    # our run, and the package run with the same settings -
    ours = L15bMethods.myintegrate(balances!, x₀, model; t_start = t_start, t_final = t_final_step, # the balances, the starting state, and the time span
        Δt = Δt, algorithm = ExplicitEulerMethod()); # the step and the method
    package = integrate(balances!, x₀, model; t_start = t_start, t_final = t_final_step, # the same balances, starting state, and time span
        Δt = Δt, algorithm = ExplicitEulerMethod()); # the package run holds one row per time

    # the times and the size of the states -
    @test ours.t == package.t; # the same grid times
    @test size(ours.x) == (length(x₀), length(package.t)); # one column per grid time
    @test ours.evaluations == package.evaluations; # the same number of rate evaluations

    # compare up to the last time at which both runs are finite. A step above the Euler limit can grow without bound -
    last_k = 0; # the last time index at which both runs are finite, found below
    for k ∈ 1:length(package.t) # each time
        if all(isfinite, package.x[k, :]) && all(isfinite, ours.x[:, k]) # both states are finite at time k
            last_k = k; # keep comparing
        else
            break; # stop at the first time at which a run is not finite
        end
    end
    @test isapprox(ours.x[:, 1:last_k], permutedims(package.x[1:last_k, :]); rtol = round_off); # the same states, with the package rows turned into columns

    # one step from the state halfway through the pulse -
    k_inside = round(Int, ((long_pulse.start + long_pulse.stop) / 2 - t_start) / Δt) + 1; # the time index halfway through the pulse
    x_inside = package.x[k_inside, :]; # the package state at that time (nmol/gDW)
    t_inside = package.t[k_inside]; # that time (min)
    @test isapprox(L15bMethods.mystep(ExplicitEulerMethod(), balances!, x_inside, model, t_inside, Δt), # our step
        advance(balances!, x_inside, model, t_inside, Δt; algorithm = ExplicitEulerMethod()); rtol = round_off, nans = true); # the package step; a NaN state gives NaN in both
end

Let's run explicit Euler and the library solve through the long pulse, and plot every protein from both runs:

In [ ]:
(euler, reference) = let

    # initialize -
    t_start = 0.0; # the time at which the runs start (min)
    edges = [long_pulse.start, long_pulse.stop]; # the pulse edges, on the grid of Δt (min)
    set_input!(model, "I", edges, [long_pulse.concentration, long_pulse.baseline]); # the long pulse, then the baseline again

    # explicit Euler at the step Δt -
    euler = L15bMethods.myintegrate(balances!, x₀, model; t_start = t_start, t_final = t_final_step, # the balances, the starting state, and the time span
        Δt = Δt, algorithm = ExplicitEulerMethod()); # one rate evaluation per step

    # the library solve, saved on the same grid, with a stop at each pulse edge -
    reference = L15bMethods.mysolve(balances!, x₀, model; t_start = t_start, t_final = t_final_step, # the same problem
        stops = edges, Δt_save = Δt, reltol = relative_tolerance, abstol = absolute_tolerance); # saved at the Euler grid times

    (euler, reference); # return
end;

plot_pulse_runs(euler, reference, model, long_pulse) # every protein from both runs, below the inducer

Compare each dashed Euler trace with the pale library trace of the same protein. Where is the gap between the two traces largest? How do you expect the gap to change if you halve `Δt`?

___
## Task 3: RK4 and the Step
In this task, we ask how fast the error of each method falls when we halve the step. RK4 evaluates the slope four times in each step and moves along a weighted average of the four slopes.

The global error is the gap between the computed state and the exact state. For small steps, the global error of a method of order $p$ is proportional to $\Delta t^{p}$, with $p = 1$ for explicit Euler and $p = 4$ for RK4.

The order holds only where the rates change smoothly. We measure the error on a window that starts at the end of the long pulse, where the inducer stays at its baseline.

Complete [the `mystep(...)` function](src/Compute.jl) for RK4, save the file, and rerun the setup cell. Let's check the RK4 step against [the `advance(...)` function](../../../code/src/FixedStepMethods.jl) and [the `integrate(...)` function](../../../code/src/FixedStepMethods.jl):

In [ ]:
let

    # initialize -
    t_start = 0.0; # the time at which the runs start (min)
    round_off = 1e-10; # the relative difference that round-off can cause between two correct runs
    edges = [long_pulse.start, long_pulse.stop]; # the pulse edges, on the grid of Δt (min)
    set_input!(model, "I", edges, [long_pulse.concentration, long_pulse.baseline]); # the long pulse, then the baseline again

    # our run, and the package run with the same settings -
    ours = L15bMethods.myintegrate(balances!, x₀, model; t_start = t_start, t_final = t_final_step, # the balances, the starting state, and the time span
        Δt = Δt, algorithm = RungeKutta4Method()); # the step and the method
    package = integrate(balances!, x₀, model; t_start = t_start, t_final = t_final_step, # the same balances, starting state, and time span
        Δt = Δt, algorithm = RungeKutta4Method()); # the package run holds one row per time

    # the times and the size of the states -
    @test ours.t == package.t; # the same grid times
    @test size(ours.x) == (length(x₀), length(package.t)); # one column per grid time
    @test ours.evaluations == package.evaluations; # the same number of rate evaluations

    # compare up to the last time at which both runs are finite. A step above the Euler limit can grow without bound -
    last_k = 0; # the last time index at which both runs are finite, found below
    for k ∈ 1:length(package.t) # each time
        if all(isfinite, package.x[k, :]) && all(isfinite, ours.x[:, k]) # both states are finite at time k
            last_k = k; # keep comparing
        else
            break; # stop at the first time at which a run is not finite
        end
    end
    @test isapprox(ours.x[:, 1:last_k], permutedims(package.x[1:last_k, :]); rtol = round_off); # the same states, with the package rows turned into columns

    # one step from the state halfway through the pulse -
    k_inside = round(Int, ((long_pulse.start + long_pulse.stop) / 2 - t_start) / Δt) + 1; # the time index halfway through the pulse
    x_inside = package.x[k_inside, :]; # the package state at that time (nmol/gDW)
    t_inside = package.t[k_inside]; # that time (min)
    @test isapprox(L15bMethods.mystep(RungeKutta4Method(), balances!, x_inside, model, t_inside, Δt), # our step
        advance(balances!, x_inside, model, t_inside, Δt; algorithm = RungeKutta4Method()); rtol = round_off, nans = true); # the package step; a NaN state gives NaN in both
end

Each entry of `step_counts` sets a grid, which splits the window into that many equal steps. Every run starts from the state of the Task 2 library solve at the end of the long pulse. On each grid, we compare protein Y from each method with a library solve saved on the same grid, and keep the largest difference.

Let's run both methods on every grid, print the rate evaluations of each run, and plot the largest error against the step:

In [ ]:
(step_sizes, euler_errors, rk4_errors) = let

    # initialize -
    window_start = long_pulse.stop; # the window starts at the end of the pulse (min)
    window_end = window_start + window_length; # the end of the window (min)
    k_start = round(Int, window_start / Δt) + 1; # the column of the Task 2 reference at the end of the pulse
    x_window = reference.x[:, k_start]; # the library state at the end of the pulse (nmol/gDW)
    step_sizes = window_length ./ step_counts; # the step of each grid (min)
    euler_errors = zeros(length(step_counts)); # the largest Euler error on each grid (nmol/gDW), filled below
    rk4_errors = zeros(length(step_counts)); # the largest RK4 error on each grid (nmol/gDW), filled below
    edges = [long_pulse.start, long_pulse.stop]; # the pulse edges; the window lies after both (min)
    set_input!(model, "I", edges, [long_pulse.concentration, long_pulse.baseline]); # the long pulse, so that the window sees the baseline

    # on each grid: the library solve, explicit Euler, and RK4, from the same state -
    for i ∈ 1:length(step_counts) # each grid
        library = L15bMethods.mysolve(balances!, x_window, model; t_start = window_start, t_final = window_end, # the window
            stops = Float64[], Δt_save = step_sizes[i], reltol = relative_tolerance, abstol = absolute_tolerance); # no stops: the inducer stays constant
        euler_run = L15bMethods.myintegrate(balances!, x_window, model; t_start = window_start, t_final = window_end, # the same window
            Δt = step_sizes[i], algorithm = ExplicitEulerMethod()); # explicit Euler on grid i
        rk4_run = L15bMethods.myintegrate(balances!, x_window, model; t_start = window_start, t_final = window_end, # the same window
            Δt = step_sizes[i], algorithm = RungeKutta4Method()); # RK4 on grid i
        euler_errors[i] = maximum(abs.(euler_run.x[protein_y, :] .- library.x[protein_y, :])); # the largest Euler error in protein Y (nmol/gDW)
        rk4_errors[i] = maximum(abs.(rk4_run.x[protein_y, :] .- library.x[protein_y, :])); # the largest RK4 error in protein Y (nmol/gDW)
        println("Step ", round(step_sizes[i]; sigdigits = 3), " min: Euler ", euler_run.evaluations, # the step and the Euler work
            ", RK4 ", rk4_run.evaluations, ", library ", library.evaluations, " rate evaluations"); # the RK4 and the library work
    end

    (step_sizes, euler_errors, rk4_errors); # return
end;

plot_error_against_step(step_sizes, euler_errors, rk4_errors) # the largest error against the step, on log-log axes

Compare the slope of each error curve with the guide lines of order one and order four. By what factor does each error fall when the step halves? Which method reaches a given error with fewer rate evaluations?

### Things to Try
* __Step above the Euler limit.__ Near a steady state, an Euler run settles only when `Δt` is below the Euler step limit, which [the example on explicit methods](../L15a/CHEME-5800-L15a-Example-ExplicitMethods-Fall-2026.ipynb) computes from the eigenvalues of [the `jacobian(...)` function](../../../code/src/GeneNetworks.jl). Compute that limit at the low steady state, set `Δt` just above it, and rerun Task 2. What does the Euler run do?
* __Change the short pulse.__ Change `short_pulse_length` and rerun Task 1. Does the run without the stops still pass over the pulse?
* __Tighten the tolerances.__ Divide `relative_tolerance` and `absolute_tolerance` by ten, and rerun Task 3. How do the rate evaluations of the library solve change?

___

## Summary
In this lab, we solved the memory circuit with a library solver, wrote the code for explicit Euler and RK4, and measured how the error of each method changes with the step.

> __Key Takeaways:__
>
> * __A library solver needs a stop at each jump of the input:__ The library solver picks its own steps, and a long step can pass over a short pulse of the input. With the pulse edges as stops, the solver ends a step at each edge.
> * __One loop runs both fixed-step methods:__ Julia picks the version of the step function by the type of its first argument. The same loop stored one column per time for explicit Euler and for RK4.
> * __The error falls with the order of the method:__ For small steps on the window after the pulse, halving the step cut the Euler error by about a factor of two and the RK4 error by about a factor of sixteen. For small errors, RK4 needed fewer rate evaluations than Euler, although each RK4 step costs four.

We can now use the same solve and the same loop with other circuits, other pulses, and other methods.

___